# Julia Lotka-Volterra Solver — Called from Python, with Gradients

This demo shows the core Tesseract value proposition:

1. A **Julia ODE solver** (using `DifferentialEquations.jl` + `SciMLSensitivity.jl`) runs inside a Tesseract container.
2. A **Python user** calls it with one function call — no Julia install needed.
3. **Gradients flow back** via SciML's adjoint sensitivity methods.
4. A simple **gradient-descent loop fits ODE parameters** to noisy observations.

## Setup

Build the Tesseract image first:

```bash
tesseract build demo/julia-diffeq/julia_diffeq
```

## What's inside the Tesseract

The container packages a Julia ODE solver that the Python side never sees directly. Here's what it does:

### The Lotka-Volterra system

The classic predator-prey model:

$$\frac{du}{dt} = \alpha u - \beta u v, \qquad \frac{dv}{dt} = \delta u v - \gamma v$$

where $u$ is the prey population, $v$ is the predator population, and $(\alpha, \beta, \delta, \gamma)$ are the four parameters we'll fit.

### Julia solver (`solve.jl`)

The solver uses [`DifferentialEquations.jl`](https://docs.sciml.ai/DiffEqDocs/stable/) with the `Tsit5()` method (Tsitouras 5th-order Runge-Kutta). The key Julia dependencies are:

| Package | Role |
|---|---|
| `DifferentialEquations.jl` | ODE solver (`Tsit5`) |
| `SciMLSensitivity.jl` | Adjoint sensitivity for gradients |
| `PythonCall.jl` / `juliacall` | In-process Julia-Python bridge |

### How gradients work

When we call `vector_jacobian_product` from Python, the Tesseract API dispatches to Julia's `adjoint_sensitivities` function, which uses SciML's **interpolating adjoint method** with `ReverseDiffVJP`:

```julia
du0_adj, dp_adj = adjoint_sensitivities(
    sol, Tsit5();
    dgdu_discrete = dgdu_discrete,
    dgdp_discrete = dgdp_discrete,
    sensealg = InterpolatingAdjoint(autojacvec=ReverseDiffVJP(true)),
    t = sol.t,
)
```

This is the same adjoint machinery that Julia users rely on natively — not finite differences, not numerical approximations. The Python consumer gets Julia-quality gradients through a single function call.

### The Tesseract API layer (`tesseract_api.py`)

The Python wrapper defines typed schemas (`InputSchema` / `OutputSchema`) with `Differentiable` annotations that tell the Tesseract runtime which fields support gradient computation. JuliaCall handles the in-process bridge — it's an implementation detail hidden inside the container.

In [ ]:
import numpy as np

from tesseract_core import Tesseract

# Ground truth parameters for the Lotka-Volterra system
TRUE_PARAMS = np.array([1.5, 1.0, 3.0, 1.0])  # alpha, beta, delta, gamma
U0 = np.array([1.0, 1.0])
T_END = 10.0
N_SAVEAT = 50

t = Tesseract.from_image("julia-diffeq")
t.serve()
print("Connected to Julia Lotka-Volterra Tesseract")
print(f"Available endpoints: {t.available_endpoints}")

## 1. Forward solve: generate synthetic observations

We solve the Lotka-Volterra system with the true parameters to get a ground-truth trajectory, then add observation noise.

In [ ]:
# Forward solve with true parameters
true_result = t.apply(
    {
        "params": TRUE_PARAMS,
        "u0": U0,
        "t_end": T_END,
        "n_saveat": N_SAVEAT,
    }
)
observed = true_result["trajectory"]
time = true_result["time"]
print(f"Forward solve: trajectory shape = {observed.shape}")

# Add observation noise
noise_std = 0.1
observed_noisy = observed + noise_std * np.random.default_rng(42).standard_normal(
    observed.shape
)

## 2. Gradient-based parameter fitting

Starting from a perturbed initial guess, we use the Tesseract's `vector_jacobian_product` endpoint to get gradients of the MSE loss with respect to the ODE parameters. SciML's adjoint sensitivity methods compute these gradients efficiently on the Julia side — from Python, it's a single function call.

In [ ]:
# Start from a perturbed initial guess
current_params = np.array([1.0, 0.5, 2.0, 0.5])
lr = 1e-4
n_steps = 100

print(f"Fitting parameters via gradient descent ({n_steps} steps)")
print(f"  True params:    {TRUE_PARAMS}")
print(f"  Initial guess:  {current_params}\n")

losses = []
param_history = [current_params.copy()]

for step in range(n_steps):
    # Forward solve
    result = t.apply(
        {
            "params": current_params,
            "u0": U0,
            "t_end": T_END,
            "n_saveat": N_SAVEAT,
        }
    )
    trajectory = result["trajectory"]

    # MSE loss and its gradient w.r.t. trajectory
    residual = trajectory - observed_noisy
    loss = float(np.mean(residual**2))
    dloss_dtraj = 2.0 * residual / residual.size
    losses.append(loss)

    # VJP: get gradient of loss w.r.t. ODE parameters
    grad = t.vector_jacobian_product(
        inputs={
            "params": current_params,
            "u0": U0,
            "t_end": T_END,
            "n_saveat": N_SAVEAT,
        },
        vjp_inputs=["params"],
        vjp_outputs=["trajectory"],
        cotangent_vector={"trajectory": dloss_dtraj},
    )

    # Gradient descent update
    current_params = current_params - lr * grad["params"]
    param_history.append(current_params.copy())

    if step % 10 == 0 or step == n_steps - 1:
        print(f"  step {step:3d}  loss={loss:.6f}  params={current_params}")

print(f"\n  Final params:   {current_params}")
print(f"  True params:    {TRUE_PARAMS}")
print(f"  Param error:    {np.linalg.norm(current_params - TRUE_PARAMS):.4f}")

## 3. Results

In [ ]:
import matplotlib.pyplot as plt

# Final forward solve with fitted parameters
final_result = t.apply(
    {
        "params": current_params,
        "u0": U0,
        "t_end": T_END,
        "n_saveat": N_SAVEAT,
    }
)
fitted = final_result["trajectory"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Trajectory comparison ---
axes[0].plot(time, observed[:, 0], "b-", label="True prey")
axes[0].plot(time, observed[:, 1], "r-", label="True predator")
axes[0].scatter(time, observed_noisy[:, 0], c="b", s=10, alpha=0.5, label="Observed")
axes[0].scatter(time, observed_noisy[:, 1], c="r", s=10, alpha=0.5)
axes[0].plot(time, fitted[:, 0], "b--", lw=2, label="Fitted prey")
axes[0].plot(time, fitted[:, 1], "r--", lw=2, label="Fitted predator")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Population")
axes[0].set_title("Lotka-Volterra: True vs Fitted")
axes[0].legend()

# --- Parameter recovery ---
axes[1].bar(
    range(4),
    TRUE_PARAMS,
    alpha=0.5,
    label="True",
    color="steelblue",
)
axes[1].bar(
    range(4),
    current_params,
    alpha=0.5,
    label="Fitted",
    color="coral",
)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels([r"$\alpha$", r"$\beta$", r"$\delta$", r"$\gamma$"])
axes[1].set_title("Parameter Recovery")
axes[1].legend()

# --- Loss curve ---
axes[2].semilogy(losses)
axes[2].set_xlabel("Step")
axes[2].set_ylabel("MSE Loss")
axes[2].set_title("Training Loss")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Clean up
t.teardown()